# ToF-SIMS mammary-tumour study — main Figures 2, 4 and 5

**Leakage-free, animal-level regeneration** of the three results figures, driven by one narrative:

1. **Figure 2 — mammary tumour is confidently classifiable.** A robust multi-ion signature separates
   tumour from healthy mammary tissue almost perfectly under leave-one-animal-out (LOAO) validation.
2. **Figure 4 — peripheral tissues.** Extending the same analysis to the three peripheral tissues, *liver*
   and *serum* are at chance, while *fur* is the only promising signal and clearly exceeds liver/serum
   (though it is not multiplicity-robust).
3. **Figure 5 — shared ions.** Mammary-signature ions are co-detected across tissues; this is descriptive
   presence, not peripheral discrimination.

**Method invariants.** The unit of analysis is the *animal*: within each LOAO fold, preprocessing
(`SpectraPreproc`: detection filter, log2, Pareto) and the classifier are fit on the training animals only,
held-out spectra are scored and averaged to one score per animal, and AUC is computed over animals. This is
the same canonical pipeline that produced `results/*.csv`; the notebook recomputes the figures from the raw
peak table so every panel is reproducible end-to-end.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.stats import mannwhitneyu
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from statsmodels.stats.multitest import multipletests
import lib_analysis as L

RES, FIG = Path("results"), Path("figures"); FIG.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 400, "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.9,
    "font.size": 10, "axes.titlesize": 12, "axes.labelsize": 10.5,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 8.5})

# Okabe-Ito colour-blind-safe palette
OI = {"cancer": "#D55E00", "control": "#0072B2", "sig": "#009E73", "ns": "#B8B8B8",
      "fur": "#CC79A7", "liver": "#E69F00", "serum": "#56B4E9",
      "pls": "#0072B2", "rf": "#009E73", "black": "#222222"}
TISSUES = ["Breast tissue", "Fur", "Liver", "Blood serum"]
LABEL = {"Breast tissue": "Mammary", "Fur": "Fur", "Liver": "Liver", "Blood serum": "Serum"}
ROBUST_CORE = [52.02, 53.02, 65.03, 66.02]   # animal-level stable core (RF & univariate freq >= 0.90)

## 1. Data and the unit of analysis

One aligned peak-area table (313 low-mass ions, *m/z* 50–250). `AnimalID` = `Group[:2] + Mouse`, so a cancer and a control mouse that share a number are distinct animals — this is what makes leave-one-**animal**-out honest.

In [ ]:
df, mass = L.load()
print(f"{len(mass)} aligned ions | {df.AnimalID.nunique()} animals | {len(df)} spectra")
overview = (df.groupby("Tissue")
            .apply(lambda g: pd.Series({"spectra": len(g), "animals": g.AnimalID.nunique(),
                    "cancer": g[g.Group=="Cancer"].AnimalID.nunique(),
                    "control": g[g.Group=="Control"].AnimalID.nunique()}))
            .loc[TISSUES].astype(int))
overview

## 2. Shared, leakage-free machinery

`pipe()` is the canonical fold-safe pipeline. `animal_ion_stats()` recomputes the animal-level Mann–Whitney table (exact for small *n*) → this is what gives **98** differential mammary ions, matching the Supplementary Information (not the 91 in the superseded `univariate_breast.csv`). Score/loading panels (PCA, PLS-DA scores, VIP, RF importance) are descriptive and fit on all spectra of the tissue.

In [ ]:
def pipe(clf, min_det=0.2):
    """Canonical leakage-free pipeline: preprocessing is fit INSIDE each fold."""
    return Pipeline([("prep", L.SpectraPreproc(min_det)), ("clf", clf)])

def animal_ion_stats(sub):
    """Animal-level two-sided Mann-Whitney per ion (unit = animal). Matches the SI."""
    am = sub.groupby("AnimalID").agg(Group=("Group", "first"),
                                     **{m: (m, "mean") for m in mass}).reset_index()
    can = am["Group"].eq("Cancer").to_numpy(); V = am[mass].to_numpy(float)
    p, rbc = [], []
    for j in range(V.shape[1]):
        a, b = V[can, j], V[~can, j]
        try:
            meth = "exact" if len(a) <= 12 and len(b) <= 12 else "asymptotic"
            r = mannwhitneyu(a, b, alternative="two-sided", method=meth)
            p.append(float(r.pvalue)); rbc.append(2.0*r.statistic/(len(a)*len(b)) - 1.0)
        except ValueError:
            p.append(1.0); rbc.append(0.0)
    fdr = multipletests(np.nan_to_num(p, nan=1.0), method="fdr_bh")[1]
    cm, ctm = V[can].mean(0), V[~can].mean(0)
    return pd.DataFrame({"Feature": np.asarray(mass, float),
                         "log2fc": np.log2((cm+1.0)/(ctm+1.0)), "rbc": rbc, "p": p, "FDR": fdr,
                         "sig": fdr < 0.05}), am

def preproc_matrix(X, min_det=0.2):
    pp = L.SpectraPreproc(min_det=min_det).fit(X)
    return pp.transform(X), np.asarray(X.columns, float)[pp.keep_]

def vip_scores(Xz, y):
    pls = PLSRegression(n_components=5, scale=False).fit(Xz, y.astype(float))
    t, w, q = pls.x_scores_, pls.x_weights_, pls.y_loadings_
    ss = (q[0]**2) * (t**2).sum(axis=0)
    vip = np.sqrt(w.shape[0] * ((w/np.linalg.norm(w, axis=0))**2 @ ss) / ss.sum())
    return vip, pls

def L_(ax, s): ax.set_title(s, loc="left", fontweight="bold", fontsize=12)
print("helpers ready")

## 3. Figure 2 — a robust mammary signature, confidently classified

Faithful 8-panel layout (A–H), recomputed animal-level:
**A** animal-level volcano (98 ions at FDR<0.05, robust core ringed) · **B** the four robust-core ions ·
**C** PCA · **D** PLS-DA scores · **E** ROC PLS-DA (LOAO) · **F** VIP · **G** RF importance ·
**H** ROC random forest (LOAO). Panels E and H are the honest performance estimates (AUC 0.997).

In [ ]:
sub = df[df.Tissue.eq("Breast tissue")].reset_index(drop=True)
Xb, yb, gb, _ = L.tissue_slice(df, mass, "Breast tissue")
stats, am = animal_ion_stats(sub)
Xz, kept = preproc_matrix(Xb)
a_pls, _ = L.animal_level_scores(Xb, yb, gb, pipe(L.PLSDAClassifier()))
a_rf,  _ = L.animal_level_scores(Xb, yb, gb, pipe(L.rf()))
perm = pd.read_csv(RES/"permutation_exact8.csv")
print(f"animal-level diff. ions (FDR<0.05): {int(stats.sig.sum())}  |  "
      f"AUC PLS {roc_auc_score(a_pls.y, a_pls.prob):.3f} / RF {roc_auc_score(a_rf.y, a_rf.prob):.3f}")

In [ ]:
fig = plt.figure(figsize=(13.2, 12.0), constrained_layout=True)
gs = GridSpec(3, 3, figure=fig)
fig.suptitle("Figure 2. A robust tumour-associated multi-ion signature confidently separates "
             "mammary tumour from healthy tissue (19 vs 19 mice)", fontsize=13, fontweight="bold")

# A volcano
ax = fig.add_subplot(gs[0,0])
ax.scatter(stats.log2fc[~stats.sig], -np.log10(stats.FDR[~stats.sig]), s=12, c=OI["ns"], edgecolors="none", label="n.s.")
ax.scatter(stats.log2fc[stats.sig],  -np.log10(stats.FDR[stats.sig]),  s=16, c=OI["sig"], edgecolors="none", label=f"FDR<0.05 (n={int(stats.sig.sum())})")
ax.axhline(-np.log10(0.05), color=OI["black"], ls="--", lw=0.8)
core = [stats.iloc[(stats.Feature-mz).abs().idxmin()] for mz in ROBUST_CORE]
ax.scatter([r.log2fc for r in core], [-np.log10(r.FDR) for r in core], s=72, facecolors="none",
           edgecolors=OI["black"], linewidths=1.5, zorder=4, label="Animal-level robust core")
ax.text(0.03, 0.03, "Robust core:\nm/z 52, 53, 65, 66", transform=ax.transAxes, fontsize=7.8, va="bottom")
ax.set(xlabel="log2 fold-change (cancer / control)", ylabel="-log10 FDR"); ax.legend(frameon=False, loc="upper left", fontsize=7.6); L_(ax, "A  Differential ions (animal-level)")

# B violins of robust core
ax = fig.add_subplot(gs[0,1])
for i, mz in enumerate(ROBUST_CORE):
    col = mass[int(np.argmin(np.abs(np.asarray(mass, float)-mz)))]
    v = np.log10(am[col].to_numpy(float)+1.0); can = am.Group.eq("Cancer").to_numpy()
    for mask, color, off in [(can, OI["cancer"], -0.18), (~can, OI["control"], 0.18)]:
        vp = ax.violinplot(v[mask], positions=[i+off], widths=0.32, showmeans=True)
        for b in vp["bodies"]: b.set_facecolor(color); b.set_alpha(0.6); b.set_edgecolor(color)
        for k in ("cbars","cmins","cmaxes","cmeans"): vp[k].set_color(color); vp[k].set_linewidth(1.0)
ax.set(xticks=range(4), xticklabels=[f"{m:.0f}" for m in ROBUST_CORE], xlabel="m/z (animal-level robust core)", ylabel="log10(mean peak area + 1)")
ax.plot([],[],color=OI["cancer"],lw=6,alpha=0.6,label="Cancer"); ax.plot([],[],color=OI["control"],lw=6,alpha=0.6,label="Control")
ax.legend(frameon=False, loc="upper right", fontsize=8); L_(ax, "B  Representative discriminative ions")

# C PCA
ax = fig.add_subplot(gs[0,2]); pcs = PCA(3, random_state=42).fit(Xz); sc = pcs.transform(Xz)
for lab, color in [("Cancer", OI["cancer"]), ("Control", OI["control"])]:
    m = (sub.Group==lab).to_numpy(); ax.scatter(sc[m,0], sc[m,1], s=14, c=color, alpha=0.6, edgecolors="none", label=lab)
ax.set(xlabel=f"PC1 ({pcs.explained_variance_ratio_[0]*100:.1f}%)", ylabel=f"PC2 ({pcs.explained_variance_ratio_[1]*100:.1f}%)"); ax.legend(frameon=False, fontsize=8); L_(ax, "C  PCA of spectra")

# D PLS-DA scores
ax = fig.add_subplot(gs[1,0]); vip, pls = vip_scores(Xz, yb); ts = pls.x_scores_
for lab, color in [("Cancer", OI["cancer"]), ("Control", OI["control"])]:
    m = (sub.Group==lab).to_numpy(); ax.scatter(ts[m,0], ts[m,1], s=14, c=color, alpha=0.6, edgecolors="none", label=lab)
ax.set(xlabel="PLS-DA LV1", ylabel="PLS-DA LV2"); ax.legend(frameon=False, fontsize=8); L_(ax, "D  PLS-DA scores")

def roc_panel(ax, a, model, letter):
    auc = roc_auc_score(a.y, a.prob); fpr, tpr, _ = roc_curve(a.y, a.prob)
    pbh = perm[(perm.Tissue.eq("Breast tissue")) & (perm.Model.eq(model))].perm_p_BH.iloc[0]
    ax.plot(fpr, tpr, color=OI["pls"] if "PLS" in model else OI["rf"], lw=2.2)
    ax.plot([0,1],[0,1], color=OI["black"], ls="--", lw=0.9)
    ax.text(0.55, 0.15, f"AUC = {auc:.3f}\nP(BH) = {pbh:.3f}", fontsize=9)
    ax.set(xlim=(-.02,1.02), ylim=(-.02,1.02), xlabel="False positive rate", ylabel="True positive rate"); L_(ax, letter)

roc_panel(fig.add_subplot(gs[1,1]), a_pls, "PLS-DA", "E  ROC - PLS-DA (LOAO)")

# F VIP
ax = fig.add_subplot(gs[1,2]); order = np.argsort(vip)[::-1][:15][::-1]
ax.barh(range(len(order)), vip[order], color=OI["pls"]); ax.axvline(1.0, color=OI["black"], ls="--", lw=0.8)
ax.set(yticks=range(len(order)), yticklabels=[f"{kept[i]:.2f}" for i in order], xlabel="VIP score", ylabel="m/z"); L_(ax, "F  PLS-DA VIP (top 15)")

# G RF importance
ax = fig.add_subplot(gs[2,0]); imp = L.rf().fit(Xz, yb).feature_importances_; order = np.argsort(imp)[::-1][:15][::-1]
ax.barh(range(len(order)), imp[order], color=OI["rf"])
ax.set(yticks=range(len(order)), yticklabels=[f"{kept[i]:.2f}" for i in order], xlabel="Random-forest importance", ylabel="m/z"); L_(ax, "G  Random-forest importance (top 15)")

roc_panel(fig.add_subplot(gs[2,1]), a_rf, "RandomForest", "H  ROC - Random forest (LOAO)")

ax = fig.add_subplot(gs[2,2]); ax.axis("off")
ax.text(0.02, 0.95, "Classification: leave-one-animal-out,\nanimal-level scoring (unit = mouse).\n\n"
        f"{int(stats.sig.sum())} ions differ at animal-level\nMann-Whitney FDR<0.05.\n\n"
        "Panels C, D, F, G are descriptive\n(fit on all spectra for display).", fontsize=9, va="top")

fig.savefig(FIG/"fig_2_mammary_signature.pdf", bbox_inches="tight")
fig.savefig(FIG/"fig_2_mammary_signature.png", bbox_inches="tight")
plt.show()

## 4. Figure 4 — the three peripheral tissues, one full panel-set each

15-panel per-tissue layout matching the manuscript caption: **serum (A–E)**, **liver (F–J)**, **fur (K–O)**,
each with PCA · PLS-DA VIP · random-forest importance · ROC PLS-DA (LOAO) · ROC random forest (LOAO). Read
top-to-bottom: serum and liver PCAs show no group structure and their ROCs sit on the diagonal (AUC ≤ 0.54),
whereas fur is the only tissue whose ROC rises above chance (0.66 / 0.74) — promising but not multiplicity-robust.

In [ ]:
row_tissues = ["Blood serum", "Liver", "Fur"]   # caption order: serum, liver, fur
letters = "ABCDEFGHIJKLMNO"
fig = plt.figure(figsize=(16.5, 9.8), constrained_layout=True); gs = GridSpec(3, 5, figure=fig)
fig.suptitle("Figure 4. Discrimination of tumour-bearing from control animals in serum, liver and fur "
             "(per-tissue, leave-one-animal-out)", fontsize=13, fontweight="bold")
for r, tissue in enumerate(row_tissues):
    X, y, g, sub = L.tissue_slice(df, mass, tissue); Xz, kept = preproc_matrix(X)
    vip, _ = vip_scores(Xz, y); imp = L.rf().fit(Xz, y).feature_importances_
    a_pls, _ = L.animal_level_scores(X, y, g, pipe(L.PLSDAClassifier()))
    a_rf,  _ = L.animal_level_scores(X, y, g, pipe(L.rf()))
    pp = {m: perm[(perm.Tissue.eq(tissue)) & (perm.Model.eq(m))].perm_p_BH.iloc[0] for m in ["PLS-DA","RandomForest"]}
    ax = fig.add_subplot(gs[r,0]); pcs = PCA(2, random_state=42).fit(Xz); sc = pcs.transform(Xz)
    for lab, color in [("Cancer", OI["cancer"]), ("Control", OI["control"])]:
        m = (sub.Group==lab).to_numpy(); ax.scatter(sc[m,0], sc[m,1], s=16, c=color, alpha=0.65, edgecolors="none", label=lab)
    ax.set(xlabel=f"PC1 ({pcs.explained_variance_ratio_[0]*100:.0f}%)", ylabel=f"PC2 ({pcs.explained_variance_ratio_[1]*100:.0f}%)")
    if r==0: ax.legend(frameon=False, fontsize=7.5)
    ax.set_title(f"{letters[r*5]}  {LABEL[tissue]} - PCA", loc="left", fontweight="bold", fontsize=11)
    ax = fig.add_subplot(gs[r,1]); o = np.argsort(vip)[::-1][:10][::-1]
    ax.barh(range(len(o)), vip[o], color=OI["pls"]); ax.axvline(1.0, color=OI["black"], ls="--", lw=0.7)
    ax.set(yticks=range(len(o)), yticklabels=[f"{kept[i]:.2f}" for i in o], xlabel="VIP"); ax.set_title(f"{letters[r*5+1]}  PLS-DA VIP", loc="left", fontweight="bold", fontsize=11)
    ax = fig.add_subplot(gs[r,2]); o = np.argsort(imp)[::-1][:10][::-1]
    ax.barh(range(len(o)), imp[o], color=OI["rf"])
    ax.set(yticks=range(len(o)), yticklabels=[f"{kept[i]:.2f}" for i in o], xlabel="RF importance"); ax.set_title(f"{letters[r*5+2]}  RF importance", loc="left", fontweight="bold", fontsize=11)
    for c, (a, model, key, color) in enumerate([(a_pls,"PLS-DA","PLS-DA",OI["pls"]), (a_rf,"RF","RandomForest",OI["rf"])]):
        ax = fig.add_subplot(gs[r,3+c]); auc = roc_auc_score(a.y, a.prob); fpr, tpr, _ = roc_curve(a.y, a.prob)
        ax.plot(fpr, tpr, color=color, lw=2.0); ax.plot([0,1],[0,1], color=OI["black"], ls="--", lw=0.8)
        ax.text(0.42, 0.12, f"AUC={auc:.2f}\nP(BH)={pp[key]:.2f}", fontsize=8)
        ax.set(xlim=(-.02,1.02), ylim=(-.02,1.02), xlabel="FPR", ylabel="TPR"); ax.set_title(f"{letters[r*5+3+c]}  ROC {model} (LOAO)", loc="left", fontweight="bold", fontsize=11)
fig.savefig(FIG/"fig_4_peripheral.pdf", bbox_inches="tight"); fig.savefig(FIG/"fig_4_peripheral.png", bbox_inches="tight")
plt.show()

## 5. Figure 5 — shared ions across tissues (descriptive)

VIP and random-forest importance of the mammary-signature ions in each tissue. **Caveat (in caption):** these
scores are *within-tissue relative*, so a bright peripheral cell does **not** mean the ion discriminates there —
no peripheral ion reached animal-level FDR<0.05 and no peripheral classifier beat chance (Figure 4). Grey =
ion not retained in that tissue.

In [ ]:
Xzb, keptb = preproc_matrix(Xb); vipb, _ = vip_scores(Xzb, yb)
top = np.sort(keptb[np.argsort(vipb)[::-1][:15]])
vip_mat = np.full((len(top), len(TISSUES)), np.nan); imp_mat = np.full_like(vip_mat, np.nan)
for ti, t in enumerate(TISSUES):
    X, y, _, _ = L.tissue_slice(df, mass, t); Xz, kept = preproc_matrix(X)
    vip, _ = vip_scores(Xz, y); imp = L.rf().fit(Xz, y).feature_importances_
    for ii, mz in enumerate(top):
        hit = np.where(np.abs(kept-mz) < 1e-6)[0]
        if hit.size: vip_mat[ii, ti] = vip[hit[0]]; imp_mat[ii, ti] = imp[hit[0]]

fig = plt.figure(figsize=(10.6, 6.6), constrained_layout=True); gs = GridSpec(1, 2, figure=fig)
fig.suptitle("Figure 5. Signature ions are co-detected across tissues, but no peripheral ion is "
             "discriminative (descriptive)", fontsize=13, fontweight="bold")
xl = [LABEL[t] for t in TISSUES]; yl = [f"{mz:.2f}" for mz in top]
for ax, mat, cbar, letter in [(fig.add_subplot(gs[0,0]), vip_mat, "VIP", "A  PLS-DA VIP"),
                              (fig.add_subplot(gs[0,1]), imp_mat, "Importance", "B  Random-forest importance")]:
    cmap = plt.cm.viridis.copy(); cmap.set_bad("#D9D9D9")
    im = ax.imshow(np.ma.masked_invalid(mat), cmap=cmap, aspect="auto")
    ax.set(xticks=range(len(xl)), yticks=range(len(yl)), yticklabels=yl, ylabel="m/z (mammary signature)")
    ax.set_xticklabels(xl, rotation=20, ha="right"); fig.colorbar(im, ax=ax, shrink=0.85, label=cbar); L_(ax, letter)
fig.text(0.5, -0.03, "Scores are within-tissue relative: a high peripheral VIP/importance does not imply "
         "discrimination. No fur/liver/serum ion reached animal-level FDR<0.05 and no peripheral\nclassifier "
         "beat chance (Figure 4); the panels show co-detection of mammary ions, not peripheral discrimination.",
         ha="center", fontsize=8.2, style="italic")
fig.savefig(FIG/"fig_5_shared_ions.pdf", bbox_inches="tight"); fig.savefig(FIG/"fig_5_shared_ions.png", bbox_inches="tight")
plt.show()

## 6. Canonical results underlying the figures

In [ ]:
summary = pd.read_csv(RES/"permutation_exact8.csv").merge(
    pd.read_csv(RES/"classification_auc.csv")[["Tissue","Model","AUC_mouse","CI_lo","CI_hi"]],
    on=["Tissue","Model"])
summary["diff_ions_FDR<0.05"] = summary["Tissue"].map(
    dict(zip(pd.read_csv(RES/"univariate_canonical.csv").Tissue, pd.read_csv(RES/"univariate_canonical.csv").sig_MWU)))
summary[["Tissue","Model","AUC_mouse","CI_lo","CI_hi","perm_p","perm_p_BH","diff_ions_FDR<0.05"]]